In [1]:
import pandas as pd
import numpy as np 
url = "https://raw.githubusercontent.com/KeithGalli/Regression-Example/master/insurance.csv"

df = pd.read_csv(url)


# find the missing values and remove it or replace it 
# Standardize the datatype into the float64 

In [2]:
display(df.info(),df.head(5),df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1272 non-null   float64
 1   sex       1272 non-null   object 
 2   bmi       1272 non-null   float64
 3   children  1272 non-null   float64
 4   smoker    1272 non-null   object 
 5   region    1272 non-null   object 
 6   charges   1284 non-null   object 
dtypes: float64(3), object(4)
memory usage: 73.3+ KB


None

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.924
1,18.0,male,33.770,1.0,no,Southeast,1725.5523
2,28.0,male,33.000,3.0,no,southeast,$4449.462
3,33.0,male,22.705,0.0,no,northwest,$21984.47061
4,32.0,male,28.880,0.0,no,northwest,$3866.8552


,age,bmi,children
count,1272.000000,1272.000000,1272.000000
mean,35.214623,30.560550,0.948899
std,22.478251,6.095573,1.303532
min,-64.000000,15.960000,-4.000000
25%,24.750000,26.180000,0.000000
50%,38.000000,30.210000,1.000000
75%,51.000000,34.485000,2.000000
max,64.000000,53.130000,5.000000


In [3]:
df.sample(10)

,age,sex,bmi,children,smoker,region,charges
143,29.0,male,29.735,2.0,no,Northwest,18157.876
558,35.0,woman,34.105,3.0,yes,northwest,39983.42595
591,47.0,male,19.570,1.0,no,northwest,8428.0693
1132,57.0,male,40.280,0.0,no,northeast,$20709.02034
909,32.0,woman,24.600,-0.0,yes,southwest,17496.306
987,45.0,female,27.645,1.0,no,northwest,28340.18885
559,19.0,male,35.530,0.0,no,Northwest,1646.4297
214,45.0,female,30.900,2.0,no,southwest,8520.026
724,50.0,female,27.075,1.0,no,northeast,10106.13425
136,19.0,male,34.100,0.0,no,southwest,1261.442


In [4]:
#convert the 
#convert age from str to numeric 
# find the out of range values  in age and replace it with nan the range should be (0,100)
# sex is [Male,female] replace all things to the binary 
#children >= 0
# replace all values to [southwest,northwest ]
#convert the charges to float 

In [5]:
df["age"] = pd.to_numeric(df["age"],errors = "coerce")
df[(df.age < 1) | (df.age > 100)] = np.nan
df.sex.replace(["male","man","M"],"male",inplace = True)
df.sex.replace(["female","woman","F"],"female",inplace = True)
df[df.children < 0] = np.nan
df["region"] = df["region"].str.lower()
df["charges"] = df["charges"].str.replace(r"[^0-9.]","",regex = True)
df["charges"] = pd.to_numeric(df["charges"],errors= "coerce")

C:\Users\sriha\AppData\Local\Temp\ipykernel_101676\323174406.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.sex.replace(["male","man","M"],"male",inplace = True)


In [6]:
df = df.dropna(how="all")
df = df.dropna(subset = ["charges"])
df["region"] = df.region.fillna(df.region.mode()[0])
df["sex"] = df.sex.fillna(df.sex.mode()[0])
df = df.dropna()
df = df.reset_index()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1110 entries, 0 to 1109
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   index     1110 non-null   int64  
 1   age       1110 non-null   float64
 2   sex       1110 non-null   object 
 3   bmi       1110 non-null   float64
 4   children  1110 non-null   float64
 5   smoker    1110 non-null   object 
 6   region    1110 non-null   object 
 7   charges   1110 non-null   float64
dtypes: float64(4), int64(1), object(3)
memory usage: 69.5+ KB


In [7]:
df.drop(["index"],inplace = True,axis = 1)

In [8]:
#regression based models and the gradient based models and distance based models requires scaling 
#Tree based models dont require scaling 
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["sex"] = label_encoder.fit_transform(df["sex"])
df["smoker"] = label_encoder.fit_transform(df["smoker"])

In [13]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop = "first")

region_encoded = ohe.fit_transform(df[["region"]])

In [17]:
df.drop(["region"],axis= 1,inplace = True)

In [ ]:
region_df = pd.DataFrame(
    region_encoded.toarray(),  
    columns=ohe.get_feature_names_out(["region"]),
    index=df.index
)
region_df


,region_northwest,region_southeast,region_southwest
0,0.0,0.0,1.0
1,0.0,1.0,0.0
2,0.0,1.0,0.0
3,1.0,0.0,0.0
4,1.0,0.0,0.0
...,...,...,...
1105,0.0,0.0,1.0
1106,1.0,0.0,0.0
1107,0.0,1.0,0.0
1108,0.0,0.0,1.0


In [23]:
final_df = pd.concat([df,region_df],axis = 1)

In [29]:
final_dfnum_cols = ["age", "bmi"]
from sklearn.preprocessing import StandardScaler


In [25]:
X = final_df.drop(columns=["charges"])
y = final_df["charges"]


In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [27]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [28]:
from sklearn.metrics import r2_score

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

r2


0.7998186834828834

In [31]:
from sklearn.preprocessing import StandardScaler
num_cols = ["age", "bmi"]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

In [32]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
r2_score(y_test, y_pred)


0.7998186834828835